## Essai Forecasting numéro 100001

In [11]:
import pickle

from IPython.core.display_functions import display
from biogeme.biogeme import BIOGEME
from biogeme.expressions import Variable, Derive, exp, Expression
from biogeme.models import lognested, nested
import biogeme.database as db
import pandas as pd

from models.logit_lmpc12_model3 import chosen_alternative as choice
from models.logit_lmpc12_model3 import database
from models.logit_lmpc12_model3 import V_3 as V
from models.logit_lmpc12_model4 import nests

In [12]:
# Nested logit model
logprob_nested = lognested(V, None, nests, choice)
biogeme_nested = BIOGEME(database, logprob_nested)
biogeme_nested.modelName = 'nested_model'

# Estimate the model
results_nested = biogeme_nested.estimate(un_bootstrap=True, recycle=True)
print(results_nested.print_general_statistics())
display(results_nested.get_estimated_parameters())

Several pickle .py are available for this model: ['nested_model.pickle', 'nested_model~00.pickle', 'nested_model~01.pickle']. The file nested_model~01.pickle is used to load the results.
Estimation results read from nested_model~01.pickle. There is no guarantee that they correspond to the specified model.


Number of estimated parameters:	14
Sample size:	5000
Excluded observations:	0
Init log likelihood:	-4167.826
Final log likelihood:	-4167.826
Likelihood ratio test for the init. model:	-0
Rho-square for the init. model:	0
Rho-square-bar for the init. model:	-0.00336
Akaike Information Criterion:	8363.651
Bayesian Information Criterion:	8454.892
Final gradient norm:	1.5966E-01
Nbr of threads:	4



,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_fare,-0.075537,0.034333,-2.200107,0.027799
beta_travel_time,-8.742886,0.981457,-8.908068,0.000000
beta_travel_time_2,-2.320468,0.604684,-3.837492,0.000124
beta_travel_time_2_non-work-education-related,0.092278,0.171094,0.539340,0.589652
beta_travel_time_3,-0.905092,0.425696,-2.126149,0.033491
beta_travel_time_3_non-work-education-related,-0.243102,0.110389,-2.202224,0.027650
beta_travel_time_4,-1.236023,0.601338,-2.055456,0.039835
beta_travel_time_4_non-work-education-related,-0.330650,0.130214,-2.539276,0.011108
beta_travel_time_non-work-education-related,0.111036,0.548394,0.202474,0.839546
constant_2,-8.618824,0.960779,-8.970658,0.000000


In [13]:
# Définir les tailles de population pour chaque segment
df = pd.read_csv('models/lpmc12.dat', sep='\t')
census = {
    'female_44_less': 2841376,
    'female_45_more': 1519948,
    'male_44_less': 2926408,
    'male_45_more': 1379198,
}

population_size = sum(census.values())

# Définir les filtres pour chaque segment
filters = {
    'female_44_less': (df['female'] == 1) & (df['age'] <= 44),
    'female_45_more': (df['female'] == 1) & (df['age'] > 44),
    'male_44_less': (df['female'] == 0) & (df['age'] <= 44),
    'male_45_more': (df['female'] == 0) & (df['age'] > 44),
}

# Count the sample size in each stratum
sample_segments = {
    segment_name: segment_rows.sum() for segment_name, segment_rows in filters.items()
}
print(f'Sample segments: {sample_segments}')

# Total sample size
total_sample = sum(sample_segments.values())
print(f'Sample size: {total_sample}')

weights = {
    segment_name: census[segment_name] * total_sample / (segment_size * population_size)
    for segment_name, segment_size in sample_segments.items()
}

# Ajouter les poids dans les datasets
for segment_name, segment_rows in filters.items():
    df.loc[segment_rows, 'weight'] = weights[segment_name]

database = db.Database("Model 4", df)

Sample segments: {'female_44_less': np.int64(1631), 'female_45_more': np.int64(1034), 'male_44_less': np.int64(1451), 'male_45_more': np.int64(884)}
Sample size: 5000


In [14]:
# Choice probabilities
prob_walk: Expression = nested(V, None, nests, 1)
prob_cycle: Expression = nested(V, None, nests, 2)
prob_pt: Expression = nested(V, None, nests, 3)
prob_car: Expression = nested(V, None, nests, 4)
# Charger les données
df['car_cost'] = (df['cost_driving_fuel'] + df['driving_traffic_percent'] * df['cost_driving_ccharge'])
# Variables
public_transport_cost = Variable('cost_transit')
car_cost = Variable('car_cost')


#direct elasticities
direct_elas_pt_cost = Derive(prob_pt, 'cost_transit') * public_transport_cost/prob_pt
direct_elas_car_cost = Derive(prob_car, 'car_cost') * car_cost / prob_car
# cross elasticities
# pt
# cros_elas_pt_cost_walk =
# cros_elas_pt_cost_cycle = 
cros_elas_pt_cost_car = Derive(prob_pt, 'car_cost') * car_cost / prob_pt
# car
# cros_elas_car_cost_walk = 
# cros_elas_car_cost_cycle = 
cros_elas_car_cost_pt = Derive(prob_car, 'cost_transit') * public_transport_cost / prob_car

In [19]:
simulate = {
    'weight': Variable('weight'),
    'Prob. public transportation': prob_pt,
    'Prob. car': prob_car,
    'direct_elas_pt_cost': direct_elas_pt_cost,
    'direct_elas_car_cost': direct_elas_car_cost,
    'cros_elas_pt_cost_car': cros_elas_pt_cost_car,
    'cros_elas_car_cost_pt': cros_elas_car_cost_pt,
}

filename = 'elasticities.pickle'
try:
    with open(filename, 'rb') as f:
        simulated_values = pickle.load(f)
        print(f'Elasticities read from {filename}')
except FileNotFoundError:
    biosim = BIOGEME(database, simulate)
    simulated_values = biosim.simulate(results_nested.get_beta_values())
    print(f'Elasticities calculated and saved in {filename}')
    with open(filename, 'wb') as f:
        pickle.dump(simulated_values, f)

Elasticities calculated and saved in elasticities.pickle


In [24]:
display(simulated_values.iloc[9])

weight                         0.900076
Prob. public transportation    0.986048
Prob. car                      0.009122
direct_elas_pt_cost            0.000000
direct_elas_car_cost           0.000000
cros_elas_pt_cost_car          0.000000
cros_elas_car_cost_pt          0.000000
Name: 9, dtype: float64

## Aggregate

In [7]:
# PT
simulated_values['numerator_pt_cost'] = (
    simulated_values['weight']
    * simulated_values['Prob. public transportation']
    * simulated_values['direct_elas_pt_cost']
)
simulated_values['denominator_pt'] = (
    simulated_values['weight'] * simulated_values['Prob. public transportation']
)
agg_elast_pt_cost = (
    simulated_values['numerator_pt_cost'].sum()
    / simulated_values['denominator_pt'].sum()
)
print(f'Aggregate elasticity wrt cost: {agg_elast_pt_cost:.3g}')

Aggregate elasticity wrt cost: -0.139


In [8]:
# CAR
simulated_values['numerator_car_cost'] = (
    simulated_values['weight']
    * simulated_values['Prob. car']
    * simulated_values['direct_elas_car_cost']
)
simulated_values['denominator_car'] = (
    simulated_values['weight'] * simulated_values['Prob. car']
)
agg_elast_car_cost = (
    simulated_values['numerator_car_cost'].sum()
    / simulated_values['denominator_car'].sum()
)
print(f'Aggregate elasticity wrt cost: {agg_elast_car_cost:.3g}')

Aggregate elasticity wrt cost: 0


In [28]:
# PT cross CAR
simulated_values['numerator_pt_cost_car'] = (
    simulated_values['weight']
    * simulated_values['Prob. public transportation']
    * simulated_values['cros_elas_pt_cost_car']
)
simulated_values['denominator_pt'] = (
    simulated_values['weight'] * simulated_values['Prob. public transportation']
)
agg_elast_pt_cost = (
    simulated_values['numerator_pt_cost_car'].sum()
    / simulated_values['denominator_pt'].sum()
)
print(f'Aggregate elasticity wrt cost: {agg_elast_pt_cost:.3g}')

Aggregate elasticity wrt cost: 0


In [29]:
# Car cross PT
simulated_values['numerator_car_cost_pt'] = (
    simulated_values['weight']
    * simulated_values['Prob. car']
    * simulated_values['cros_elas_car_cost_pt']
)
simulated_values['denominator_car'] = (
    simulated_values['weight'] * simulated_values['Prob. car']
)
agg_elast_car_cost = (
    simulated_values['numerator_car_cost_pt'].sum()
    / simulated_values['denominator_car'].sum()
)
print(f'Aggregate elasticity wrt cost: {agg_elast_car_cost:.3g}')

Aggregate elasticity wrt cost: 0.108
